In [0]:
from pyspark.sql import functions as F
df = spark.table("echochain.bronze.raw_listings")

In [0]:
df.head()

Row(bids=3, brand='Dell', condition='For parts or not working', listing_id='v1|203847561029|0', listing_url='https://www.ebay.com/itm/203847561029', location='Austin, TX', model='Latitude 5420', price=189.99, processor='Intel Core i5-1145G7', ram_gb=16, scraped_at='2026-08-20T11:13:45.300577+00:00', screen_size_in='14 in', seller_notes='Motherboard boots fine, display has visible crack, otherwise good condition. Battery holds charge.', shipping_price=15.5, ssd_gb=256, title='Dell Latitude 5420 14in Laptop i5-1145G7 16GB 256GB SSD - CRACKED SCREEN', watchers=12)

In [0]:
#text normalization - lowercase, trim : title, brand, model
df = df.withColumn("title_clean", F.trim(F.lower(F.col("title")))) \
       .withColumn("brand_clean", F.trim(F.lower(F.col("brand")))) \
       .withColumn("model_clean", F.trim(F.lower(F.col("model"))))

In [0]:
df.head()

Row(bids=3, brand='Dell', condition='For parts or not working', listing_id='v1|203847561029|0', listing_url='https://www.ebay.com/itm/203847561029', location='Austin, TX', model='Latitude 5420', price=189.99, processor='Intel Core i5-1145G7', ram_gb=16, scraped_at='2026-08-20T11:13:45.300577+00:00', screen_size_in='14 in', seller_notes='Motherboard boots fine, display has visible crack, otherwise good condition. Battery holds charge.', shipping_price=15.5, ssd_gb=256, title='Dell Latitude 5420 14in Laptop i5-1145G7 16GB 256GB SSD - CRACKED SCREEN', watchers=12, title_clean='dell latitude 5420 14in laptop i5-1145g7 16gb 256gb ssd - cracked screen', brand_clean='dell', model_clean='latitude 5420')

In [0]:
#backfilling RAM from title for null scenarios
#using regex function - excludes numbers followed by "SSD"/"NVMe", these are for storage
ram_regex = r"(\d+)\s*GB(?!\s*(?:SSD|NVMe))"
 
df = df.withColumn(
    "ram_gb_extracted",
    F.regexp_extract(F.col("title"), ram_regex, 1)
).withColumn(
    "ram_gb_final",
    F.coalesce(
        F.col("ram_gb"),
        F.when(F.col("ram_gb_extracted") != "", F.col("ram_gb_extracted").cast("int"))
    )
)

In [0]:
#backfilling SSD from title for null scenarios
#using regex function - Requires "SSD" or "NVMe" immediately after the number
storage_regex = r"(\d+)\s*GB\s*(?:SSD|NVMe)"
 
df = df.withColumn(
    "ssd_gb_extracted",
    F.regexp_extract(F.col("title"), storage_regex, 1)
).withColumn(
    "ssd_gb_final",
    F.coalesce(
        F.col("ssd_gb"),
        F.when(F.col("ssd_gb_extracted") != "", F.col("ssd_gb_extracted").cast("int"))
    )
)

In [0]:
#backfilling processor from title for null scenarios
#Covers Intel (i5-1145G7 style) and AMD Ryzen patterns
#only fills in what the title actually contains - if the title just says "Core i7" with no model number, this correctly returns null rather than guessing.
cpu_regex = r"(i[3579]-\d{3,5}[A-Z0-9]*|Ryzen\s+\d\s*(?:PRO)?\s*\d{4}[A-Z]*)"
 
df = df.withColumn(
    "processor_extracted",
    F.regexp_extract(F.col("title"), cpu_regex, 1)
).withColumn(
    "processor_final",
    F.coalesce(
        F.col("processor"),
        F.when(F.col("processor_extracted") != "", F.col("processor_extracted"))
    )
)

In [0]:
#damage_flag column creation, from title and seller_notes 
#gives a clearer picture on the condition and usability of the product
#if none of the regex words match, then it fills it as unknown
damage_regex = r"(?i)(crack|broken|hinge|dead pixel|for parts|spares|as is|repair|flicker|no boot|not work)"
 
df = df.withColumn(
    "combined_text",
    F.concat_ws(" ", F.col("title"), F.coalesce(F.col("seller_notes"), F.lit("")))
).withColumn(
    "damage_flag",
    F.when(F.col("combined_text").rlike(damage_regex), F.lit("likely_damaged")).otherwise(F.lit("unknown"))
)

In [0]:
#product_signature column, this col will be used to fuzzy match against the SKUs in BOM data
#combines brand + model + ram(if not null) + ssd(if not null)
df = df.withColumn(
    "ram_token",
    F.when(F.col("ram_gb_final").isNotNull(), F.concat(F.col("ram_gb_final").cast("string"), F.lit("gb")))
).withColumn(
    "ssd_token",
    F.when(F.col("ssd_gb_final").isNotNull(), F.concat(F.col("ssd_gb_final").cast("string"), F.lit("gb")))
).withColumn(
    "product_signature",
    F.trim(F.concat_ws(" ", F.col("brand_clean"), F.col("model_clean"), F.col("ram_token"), F.col("ssd_token")))
)

In [0]:
#Selecting the final column set and storing in Bronze schema as cleaned_listings
listings_cleaned = df.select(
    "listing_id",
    "listing_url",
    "title_clean",
    "seller_notes",
    "condition",
    "damage_flag",
    "price",
    "shipping_price",
    "bids",
    "watchers",
    "location",
    "brand_clean",
    "model_clean",
    "processor_final",
    F.col("ram_gb_final").alias("ram_gb"),
    F.col("ssd_gb_final").alias("ssd_gb"),
    "screen_size_in",
    "product_signature",
    "scraped_at",
)
 
listings_cleaned.write.format("delta").mode("overwrite").saveAsTable("echochain.bronze.cleaned_listings")
display(listings_cleaned)

listing_id,listing_url,title_clean,seller_notes,condition,damage_flag,price,shipping_price,bids,watchers,location,brand_clean,model_clean,processor_final,ram_gb,ssd_gb,screen_size_in,product_signature,scraped_at
v1|203847561029|0,https://www.ebay.com/itm/203847561029,dell latitude 5420 14in laptop i5-1145g7 16gb 256gb ssd - cracked screen,"Motherboard boots fine, display has visible crack, otherwise good condition. Battery holds charge.",For parts or not working,likely_damaged,189.99,15.5,3,12,"Austin, TX",dell,latitude 5420,Intel Core i5-1145G7,16,256,14 in,dell latitude 5420 16gb 256gb,2026-08-20T11:13:45.300577+00:00
v1|198822739104|0,https://www.ebay.com/itm/198822739104,dell latitude 5420 - excellent - battery new - fast shipping,null,Used,unknown,312.0,0.0,0,4,"Reno, NV",dell,latitude 5420,null,16,256,null,dell latitude 5420 16gb 256gb,2026-08-20T11:13:45.300577+00:00
v1|204471190833|0,https://www.ebay.com/itm/204471190833,hp elitebook 840 g7 core i7 16gb ram 512gb ssd backlit keyboard,"Light scratches on lid, screen is perfect. Includes original charger.",Used,unknown,278.5,12.0,7,21,"Columbus, OH",hp,elitebook 840 g7,Intel Core i7-10510U,16,512,null,hp elitebook 840 g7 16gb 512gb,2026-08-20T11:13:45.302589+00:00
v1|201938847215|0,https://www.ebay.com/itm/201938847215,hp elitebook 840g7 spares or repair no charger no boot,"Selling as is, does not power on, sold for parts only, no returns",For parts or not working,likely_damaged,65.0,18.75,1,2,"Leeds, United Kingdom",hp,elitebook 840 g7,null,null,null,null,hp elitebook 840 g7,2026-08-20T11:13:45.302589+00:00
v1|206612348890|0,https://www.ebay.com/itm/206612348890,lenovo thinkpad t14 gen 1 amd ryzen 7 16gb 512gb nvme win11 pro,"Tested working, minor wear on palm rest, battery holds ~85% capacity",Used,unknown,341.25,0.0,0,9,"Denver, CO",lenovo,thinkpad t14 gen 1,AMD Ryzen 7 PRO 4750U,16,512,14 in,lenovo thinkpad t14 gen 1 16gb 512gb,2026-08-20T11:13:45.304598+00:00
v1|207743102256|0,https://www.ebay.com/itm/207743102256,lenovo thinkpad t14 great condition fast laptop office student,null,Used,unknown,299.99,9.99,2,6,"Newark, NJ",lenovo,null,null,16,null,null,lenovo 16gb,2026-08-20T11:13:45.305105+00:00
v1|209981223470|0,https://www.ebay.com/itm/209981223470,dell latitude 5420 laptop - as is - hinge broken screen flickers,"Sold as-is for parts or repair, motherboard untested, hinge cracked, screen flickers intermittently",For parts or not working,likely_damaged,54.99,14.2,4,15,"Phoenix, AZ",dell,latitude 5420,Intel Core i5-1145G7,null,256,null,dell latitude 5420 256gb,2026-08-20T11:13:45.306111+00:00
v1|210456781902|0,https://www.ebay.com/itm/210456781902,hp elitebook 840 g7 i7 16/512 business laptop refurbished grade a,"Professionally refurbished, new battery installed, 90-day warranty included",Certified - Refurbished,unknown,399.0,0.0,0,18,"Miami, FL",hp,elitebook 840 g7,Intel Core i7-10510U,16,512,14 in,hp elitebook 840 g7 16gb 512gb,2026-08-20T11:13:45.308101+00:00
v1|211029384756|0,https://www.ebay.com/itm/211029384756,lenovo t14 ryzen laptop good battery life screen has dead pixels,"Works great, small cluster of dead pixels top-left corner, barely noticeable",Used,likely_damaged,215.0,11.4,1,3,"Toronto, ON, Canada",lenovo,thinkpad t14 gen 1,AMD Ryzen 7 PRO 4750U,16,null,null,lenovo thinkpad t14 gen 1 16gb,2026-08-20T11:13:45.310108+00:00
v1|212398761045|0,https://www.ebay.com/itm/212398761045,dell latitude 5420 bundle lot of 2 - both power on - read description,"Lot of 2 units, both power on and boot to BIOS, no OS installed, sold together only",Used,unknown,240.0,22.0,0,7,"Sacramento, CA",dell,latitude 5420,null,null,null,null,dell latitude 5420,2026-08-20T11:13:45.311196+00:00
